# Part 2: Image Preprocessing and Augmentation

In this module, we will load the images using OpenCV, convert them to normalized grayscale tensors, split the dataset, and prepare a data augmentation pipeline.

In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight


In [ ]:
images = []
labels = []
IMG_SIZE = 64
failed_count = 0

for image_name in os.listdir(DATASET_DIR):
    if not image_name.endswith(('.png', '.jpg', '.jpeg')):
        continue
    parts = image_name.split('_')
    if len(parts) < 2:
        continue
    class_name = f"{parts[0]}_{parts[1]}"
    if class_name not in valid_classes:
        continue
        
    img_path = os.path.join(DATASET_DIR, image_name)
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img is None:
        failed_count += 1
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    images.append(img)
    labels.append(label_map[class_name])

print(f"Images Loaded: {len(images)}")
print(f"Failed to load: {failed_count}")

In [ ]:
from tensorflow.keras.utils import to_categorical

X = np.array(images).reshape(-1, IMG_SIZE, IMG_SIZE, 3)
y = to_categorical(np.array(labels))

print("X Shape:", X.shape)
print("y Shape:", y.shape)

In [ ]:
# Create an 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Training Set Shape:", X_train.shape)
print("Validation Set Shape :", X_test.shape)

In [ ]:
# Compute class weights for balancing
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(np.argmax(y_train, axis=1)),
    y=np.argmax(y_train, axis=1)
)
class_weight_dict = dict(enumerate(class_weights))
print("Computed class weights!")

datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.2,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

datagen.fit(X_train)
print("Data Augmentation configured.")